In [ ]:
# Cell 1: imports, paths, constants

import os
import shutil
from pathlib import Path

import numpy as np
from tqdm.auto import tqdm
from PIL import Image

# Paths
REPO_ROOT = Path.cwd().parents[0] if Path.cwd().name == "notebooks" else Path.cwd()

RAW_ROOT = REPO_ROOT / "data" / "raw" / "pcb defect.v1i.yolov5pytorch"
SW_PATCH_ROOT = REPO_ROOT / "data" / "patches_sw"

print("Repo root:", REPO_ROOT)
print("RAW_ROOT :", RAW_ROOT)
print("SW_PATCH_ROOT:", SW_PATCH_ROOT)

# Classes & sliding-window params
DEFECT_CLASSES = [
    "missing_hole",
    "mouse_bite",
    "open_circuit",
    "short",
    "spur",
    "spurious_copper",
]
ALL_CLASSES = ["background"] + DEFECT_CLASSES

PATCH_SIZE = 128
STRIDE = 64           
BG_KEEP_PROB = 0.2      


In [ ]:
# Cell 2: YOLO label reader + helpers

def read_yolo_labels(label_path, img_w, img_h):
    """
    Return a list of (cls_idx, x1, y1, x2, y2) in PIXELS from a YOLO txt file.
    YOLO format per line: cls cx cy w h (all normalized 0–1).
    """
    boxes = []
    if not label_path.exists():
        return boxes

    with open(label_path, "r") as f:
        for line in f:
            parts = line.strip().split()
            if len(parts) != 5:
                continue
            cls = int(parts[0])
            cx, cy, bw, bh = map(float, parts[1:])

            # convert to pixel coords
            cx *= img_w
            cy *= img_h
            bw *= img_w
            bh *= img_h

            x1 = cx - bw / 2
            y1 = cy - bh / 2
            x2 = cx + bw / 2
            y2 = cy + bh / 2

            # clip
            x1 = max(0, min(img_w - 1, x1))
            y1 = max(0, min(img_h - 1, y1))
            x2 = max(0, min(img_w - 1, x2))
            y2 = max(0, min(img_h - 1, y2))

            boxes.append((cls, x1, y1, x2, y2))
    return boxes


def box_iou_xyxy(box_a, box_b):
    """IoU between box_a and box_b, each = (x1,y1,x2,y2)."""
    xa1, ya1, xa2, ya2 = box_a
    xb1, yb1, xb2, yb2 = box_b

    inter_x1 = max(xa1, xb1)
    inter_y1 = max(ya1, yb1)
    inter_x2 = min(xa2, xb2)
    inter_y2 = min(ya2, yb2)

    if inter_x2 <= inter_x1 or inter_y2 <= inter_y1:
        return 0.0

    inter_area = (inter_x2 - inter_x1) * (inter_y2 - inter_y1)
    area_a = (xa2 - xa1) * (ya2 - ya1)
    area_b = (xb2 - xb1) * (yb2 - yb1)

    return inter_area / (area_a + area_b - inter_area + 1e-9)


In [ ]:
# Cell 3: overlap-based labeling (used in final version)

def label_window_overlap(x1, y1, x2, y2, gt_boxes, min_rel_overlap=0.30):
    """
    Label a sliding window based on how much it overlaps any ground-truth box.

    - gt_boxes: list of (cls_idx, gx1, gy1, gx2, gy2) in pixels.
    - min_rel_overlap: minimum (intersection_area / box_area) to mark as positive.

    Returns: class_name ("missing_hole", ..., "background").
    """

    best_cls = None
    best_ratio = 0.0

    for (cls_idx, gx1, gy1, gx2, gy2) in gt_boxes:
        # intersection
        ix1 = max(x1, gx1)
        iy1 = max(y1, gy1)
        ix2 = min(x2, gx2)
        iy2 = min(y2, gy2)

        if ix2 <= ix1 or iy2 <= iy1:
            continue  # no overlap

        inter_area = (ix2 - ix1) * (iy2 - iy1)
        box_area   = (gx2 - gx1) * (gy2 - gy1)
        if box_area <= 0:
            continue

        # how much of the *defect box* is covered by this window?
        rel_overlap = inter_area / box_area

        if rel_overlap > best_ratio:
            best_ratio = rel_overlap
            best_cls = DEFECT_CLASSES[cls_idx]

    if best_cls is not None and best_ratio >= min_rel_overlap:
        return best_cls

    return "background"


In [ ]:
# Cell 4: generate patches for one image

def generate_patches_for_image(img_path, split):
    """
    Slide window over single image, label each patch via overlap-with-box rule, save cropped patch.
    Returns: (num_pos, num_bg)
    """
    img = Image.open(img_path).convert("RGB")
    W, H = img.size

    # label file path
    label_path = img_path.parent.parent / "labels" / (img_path.stem + ".txt")

    if not label_path.exists():
        print(f"[WARN] No label file for: {img_path.name}")
        gt_boxes = []
    else:
        gt_boxes = read_yolo_labels(label_path, W, H)

    num_pos = 0
    num_bg = 0

    base_name = img_path.stem

    for y in range(0, H - PATCH_SIZE + 1, STRIDE):
        for x in range(0, W - PATCH_SIZE + 1, STRIDE):
            x1, y1 = x, y
            x2, y2 = x + PATCH_SIZE, y + PATCH_SIZE

            cls_name = label_window_overlap(
                x1, y1, x2, y2, gt_boxes,
                min_rel_overlap=0.30
            )

            # background subsampling
            if cls_name == "background" and np.random.rand() > BG_KEEP_PROB:
                continue

            patch = img.crop((x1, y1, x2, y2))

            out_dir = SW_PATCH_ROOT / split / cls_name
            out_name = f"{base_name}_x{x}_y{y}.jpg"
            patch.save(out_dir / out_name, quality=95)

            if cls_name == "background":
                num_bg += 1
            else:
                num_pos += 1

    return num_pos, num_bg


In [ ]:
# Cell 5: reset patch directory structure

if SW_PATCH_ROOT.exists():
    print("Removing old sliding-window patches at:", SW_PATCH_ROOT)
    shutil.rmtree(SW_PATCH_ROOT)

for split in ["train", "valid", "test"]:
    for cls in ALL_CLASSES:
        out_dir = SW_PATCH_ROOT / split / cls
        out_dir.mkdir(parents=True, exist_ok=True)

print("Fresh SW_PATCH_ROOT created:", SW_PATCH_ROOT)


In [ ]:
# Cell 6: test on a single train image to check counts

train_img_dir = RAW_ROOT / "train" / "images"
img_list = sorted(train_img_dir.glob("*.jpg"))

if not img_list:
    print("No .jpg files found in:", train_img_dir)
else:
    one_img = img_list[0]
    print("Test image:", one_img.name)

    p, b = generate_patches_for_image(one_img, split="train")
    print("From this single image -> positives:", p, "background:", b)


In [ ]:
# Cell 6: test on a single train image to check counts

train_img_dir = RAW_ROOT / "train" / "images"
one_img = sorted(train_img_dir.glob("*.jpg"))[0]
print("Test image:", one_img.name)

p, b = generate_patches_for_image(one_img, split="train")
print("From this single image -> positives:", p, "background:", b)


In [ ]:
# Cell 7: generate all patches for train/valid/test

def generate_all_patches_for_split(split):
    img_dir = RAW_ROOT / split / "images"
    img_paths = sorted(img_dir.glob("*.jpg"))

    total_pos = 0
    total_bg = 0

    print(f"\n=== Generating sliding-window patches for {split} "
          f"({len(img_paths)} images) ===")
    for img_path in tqdm(img_paths):
        p, b = generate_patches_for_image(img_path, split)
        total_pos += p
        total_bg += b

    print(f"Split {split}: positives={total_pos}, background_kept={total_bg}")


for split in ["train", "valid", "test"]:
    generate_all_patches_for_split(split)


In [ ]:
# Cell 8: per-class patch counts (should match your earlier numbers up to RNG)

PRINT_ROOT = SW_PATCH_ROOT

for split in ["train", "valid", "test"]:
    print("\n", split)
    split_dir = PRINT_ROOT / split
    for cls in sorted(os.listdir(split_dir)):
        cls_dir = split_dir / cls
        if cls_dir.is_dir():
            n = len(list(cls_dir.glob("*.jpg")))
            print(f"  {cls:15s} -> {n}")
